# M5.S1 Hybrid computing

We will calculate how heat spreads through a plate, then run the same calculation with threads, processes and a GPU.

Each version has its own C file. We change one idea at a time: first the stencil, then OpenMP, then MPI, then the combination. There are no `#ifdef` switches and no supporting build framework.

You can work in SSH or use Jupyter. The calculation is always the C program. Jupyter only writes the files and runs the same shell commands that you would type in SSH.

## 1. What does the solver calculate?

We hold the top edge of a square plate at 100 °C. The other edges and the initial interior are 0 °C. `N` includes the boundary cells.

The heat equation is

$$\frac{\partial T}{\partial t}=\alpha\left(\frac{\partial^2 T}{\partial x^2}+\frac{\partial^2 T}{\partial y^2}\right).$$

On an equally spaced grid, a simple explicit update is

$$T^{new}_{i,j}=T^{old}_{i,j}+r\left(T^{old}_{i-1,j}+T^{old}_{i+1,j}+T^{old}_{i,j-1}+T^{old}_{i,j+1}-4T^{old}_{i,j}\right),\qquad r=\frac{\alpha\Delta t}{h^2}.$$

We choose $h=1$, $\alpha=1$ and $\Delta t=0.25$, giving $r=0.25$. The new temperature is then the average of the four neighbours. This choice is at the stability limit for this method in two dimensions; increasing the time step would make it unstable.

The line to find in every program is:

```c
next[k] = 0.25*(old[k-n] + old[k+n] + old[k-1] + old[k+1]);
```

We need two arrays because every update must read the old time level. In a flat array, `(y,x)` is at `y*n+x`. Adding or subtracting `n` moves one row. Adding or subtracting one moves one column.

After one step, an interior cell immediately below the hot edge should be 25 °C. That is our first independent check.

## 2. Prepare your folder

In SSH, replace `YOUR_USERNAME` with your account:

```bash
ssh YOUR_USERNAME@ssh.iesci.tech
mkdir -p ~/hpc-labs/m5s1-basics
cd ~/hpc-labs/m5s1-basics
pwd
hostname
module avail OpenMPI
module load OpenMPI/4.1.5-GCC-12.3.0
module list
which gcc
which mpicc
```

A module changes the compiler/library paths for your shell. It does not install software in your folder. This module name was available on SciTech during preparation; inspect the catalogue if it changes.

### Checkpoint 1

Confirm your folder and compiler paths. Record the hostname and module in `observations.txt`.

In Jupyter, the next cell selects the same folder. A cell beginning `%%writefile` saves the C or shell text below it. In SSH, use `nano filename` and paste that text, leaving out the `%%writefile` line. Neither operation compiles or runs a solver.

In [ ]:
from pathlib import Path
import os

folder = Path.home() / 'hpc-labs' / 'm5s1-basics'
folder.mkdir(parents=True, exist_ok=True)
os.chdir(folder)
print('Working folder:', folder)

## 3. Start with one CPU core

Read `update` first. It contains the numerical calculation. Then read the loop in `main`: it alternates the two arrays so that we do not copy the entire plate after every step.

`seconds` measures elapsed time. `save_grid` only writes the result in VTK format so ParaView can open it directly. These functions do not change the heat calculation.

In [ ]:
%%writefile heat_serial.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

double seconds(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + 1e-9*t.tv_nsec;
}

/* Read only the old grid; each iteration writes a different new cell. */
void update(double *old, double *next, int n) {
    for (int y = 1; y < n-1; y++) {
        for (int x = 1; x < n-1; x++) {
            int k = y*n+x;
            next[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
        }
    }
}

void save_grid(const char *name, const double *t, int n);

int main(int argc, char **argv) {
    if (argc != 4) { fprintf(stderr, "Usage: %s N STEPS output.vtk\n", argv[0]); return 1; }
    int n = atoi(argv[1]), steps = atoi(argv[2]);
    if (n < 3 || n > 4096 || steps < 1 || steps > 100000) return 1;
    double *a = calloc((size_t)n*n, sizeof(double));
    double *b = calloc((size_t)n*n, sizeof(double));
    if (!a || !b) return 1;
    /* Both grids start cold, with the same fixed hot top edge. */
    for (int x = 0; x < n; x++) a[x] = b[x] = 100.0;
    {
        double start = seconds();
        for (int step = 0; step < steps; step++) {
            /* Alternate buffers; no whole-grid copy is needed. */
            double *old = step % 2 == 0 ? a : b;
            double *next = step % 2 == 0 ? b : a;
            update(old, next, n);
        }
        printf("serial loop_s=%.9f\n", seconds()-start);
    }
    double *final = steps % 2 == 0 ? a : b;
    printf("Cell below hot edge: %.9f C\n", final[n+1]);
    save_grid(argv[3], final, n);
    free(a); free(b);
    return 0;
}

/* VTK adds grid dimensions so ParaView can open the result directly. */
void save_grid(const char *name, const double *t, int n) {
    FILE *f = fopen(name, "w");
    if (!f) { perror(name); exit(1); }
    fprintf(f, "# vtk DataFile Version 3.0\nHeat plate, Celsius\nASCII\n");
    fprintf(f, "DATASET STRUCTURED_POINTS\nDIMENSIONS %d %d 1\n", n, n);
    fprintf(f, "ORIGIN 0 0 0\nSPACING 1 1 1\nPOINT_DATA %d\n", n*n);
    fprintf(f, "SCALARS Temperature double 1\nLOOKUP_TABLE default\n");
    for (int k = 0; k < n*n; k++) fprintf(f, "%.17g\n", t[k]);
    if (fclose(f) != 0) exit(1);
}

### Compile and run

Type these commands in SSH from the lab folder. Compilation can happen on the login node; `srun` requests a compute core before executing the solver.

```bash
module load OpenMPI/4.1.5-GCC-12.3.0
gcc -O2 heat_serial.c -o serial
srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:02:00 ./serial 8 1 first.vtk
```

`8 1` means an 8 by 8 grid and one time step. The final argument is the output file. The program prints the update-loop time and the temperature of a cell below the hot edge.

### Checkpoint 2

The printed temperature must be 25 °C. The VTK file has ten header lines followed by one temperature per line. Show the first interior row:

```bash
sed -n '19,26p' first.vtk
```

Expect `0 25 25 25 25 25 25 0`, one value per line. The two zeros are the fixed side edges. Now run a longer calculation and keep its output:

```bash
srun -p cpu -N1 -n1 -c1 --mem=1G --time=00:02:00 ./serial 128 150 serial.vtk | tee serial-time.txt
```

Keep the grid size and step count the same when comparing implementations.

## 4. Share the work with OpenMP

This is a separate, complete program. The important addition is `#pragma omp parallel for` just before the row loop in `update`. Threads read the same old array and write different new cells. The barrier at the end of the loop finishes this time step before the next starts.

The rest of the solver is the same. Look at the two files side by side and find the added header, directive and thread information.

In [ ]:
%%writefile heat_openmp.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <omp.h>

double seconds(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + 1e-9*t.tv_nsec;
}

/* Read only the old grid; each iteration writes a different new cell. */
void update(double *old, double *next, int n) {
    #pragma omp parallel for
    for (int y = 1; y < n-1; y++) {
        for (int x = 1; x < n-1; x++) {
            int k = y*n+x;
            next[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
        }
    }
}

void save_grid(const char *name, const double *t, int n);

int main(int argc, char **argv) {
    if (argc != 4) { fprintf(stderr, "Usage: %s N STEPS output.vtk\n", argv[0]); return 1; }
    int n = atoi(argv[1]), steps = atoi(argv[2]);
    if (n < 3 || n > 4096 || steps < 1 || steps > 100000) return 1;
    double *a = calloc((size_t)n*n, sizeof(double));
    double *b = calloc((size_t)n*n, sizeof(double));
    if (!a || !b) return 1;
    /* Both grids start cold, with the same fixed hot top edge. */
    for (int x = 0; x < n; x++) a[x] = b[x] = 100.0;
    printf("OpenMP thread limit: %d\n", omp_get_max_threads());
    {
        double start = seconds();
        for (int step = 0; step < steps; step++) {
            /* Alternate buffers; no whole-grid copy is needed. */
            double *old = step % 2 == 0 ? a : b;
            double *next = step % 2 == 0 ? b : a;
            update(old, next, n);
        }
        printf("openmp loop_s=%.9f\n", seconds()-start);
    }
    double *final = steps % 2 == 0 ? a : b;
    printf("Cell below hot edge: %.9f C\n", final[n+1]);
    save_grid(argv[3], final, n);
    free(a); free(b);
    return 0;
}

/* VTK adds grid dimensions so ParaView can open the result directly. */
void save_grid(const char *name, const double *t, int n) {
    FILE *f = fopen(name, "w");
    if (!f) { perror(name); exit(1); }
    fprintf(f, "# vtk DataFile Version 3.0\nHeat plate, Celsius\nASCII\n");
    fprintf(f, "DATASET STRUCTURED_POINTS\nDIMENSIONS %d %d 1\n", n, n);
    fprintf(f, "ORIGIN 0 0 0\nSPACING 1 1 1\nPOINT_DATA %d\n", n*n);
    fprintf(f, "SCALARS Temperature double 1\nLOOKUP_TABLE default\n");
    for (int k = 0; k < n*n; k++) fprintf(f, "%.17g\n", t[k]);
    if (fclose(f) != 0) exit(1);
}

### Compile, allocate two cores and compare

```bash
gcc -O2 -fopenmp heat_openmp.c -o openmp
export OMP_NUM_THREADS=2
export OMP_DYNAMIC=FALSE
srun -p cpu -N1 -n1 -c2 --mem=1G --time=00:02:00 ./openmp 128 150 openmp.vtk | tee openmp-time.txt
cmp serial.vtk openmp.vtk
```

`-fopenmp` enables the OpenMP directive. `-c2` requests two cores for one process. `OMP_NUM_THREADS=2` asks that process to use two threads.

### Checkpoint 3

`cmp` should print nothing and return successfully. The complete text files, including all temperatures, are identical. If they differ, stop before discussing speed.

Is OpenMP faster for this small plate? Compare the printed loop times. Starting and synchronising threads has a cost, so two threads need not win. Repeat the runs before drawing a conclusion.

## 5. Give different rows to MPI processes

MPI processes do not share arrays. Each rank owns some interior rows and two halo rows. It exchanges its first and last owned rows with its neighbours before applying the same stencil.

To keep the example short, use a rank count that divides `N-2`. For `N=128` and two ranks, each owns 63 interior rows. Rank 0 holds the physical hot edge in its upper halo. `MPI_PROC_NULL` means there is no neighbour beyond a physical edge.

Read the two `MPI_Sendrecv` calls and the update loop. The final `MPI_Gather` collects the plate once, after all time steps.

In [ ]:
%%writefile heat_mpi.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>

/* VTK adds grid dimensions so ParaView can open the result directly. */
void save_grid(const char *name, const double *t, int n) {
    FILE *f = fopen(name, "w");
    if (!f) { perror(name); exit(1); }
    fprintf(f, "# vtk DataFile Version 3.0\nHeat plate, Celsius\nASCII\n");
    fprintf(f, "DATASET STRUCTURED_POINTS\nDIMENSIONS %d %d 1\n", n, n);
    fprintf(f, "ORIGIN 0 0 0\nSPACING 1 1 1\nPOINT_DATA %d\n", n*n);
    fprintf(f, "SCALARS Temperature double 1\nLOOKUP_TABLE default\n");
    for (int k = 0; k < n*n; k++) fprintf(f, "%.17g\n", t[k]);
    if (fclose(f) != 0) exit(1);
}

int main(int argc, char **argv) {
    int rank, ranks;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &ranks);
    if (argc != 4) MPI_Abort(MPI_COMM_WORLD, 1);
    int n = atoi(argv[1]), steps = atoi(argv[2]);
    /* Equal blocks keep this example short: N-2 must divide by ranks. */
    if (n < 3 || n > 4096 || steps < 1 || steps > 100000 || ranks > n-2 || (n-2)%ranks != 0)
        MPI_Abort(MPI_COMM_WORLD, 1);
    int rows = (n-2)/ranks;
    double *old = calloc((size_t)(rows+2)*n, sizeof(double));
    double *next = calloc((size_t)(rows+2)*n, sizeof(double));
    if (!old || !next) MPI_Abort(MPI_COMM_WORLD, 1);
    /* Each rank has two halo rows. Rank 0's top halo is the hot edge. */
    if (rank == 0) for (int x = 0; x < n; x++) old[x] = next[x] = 100.0;
    int up = rank == 0 ? MPI_PROC_NULL : rank-1;
    int down = rank == ranks-1 ? MPI_PROC_NULL : rank+1;
    fprintf(stderr, "rank %d/%d: %d interior rows\n", rank, ranks, rows);
    MPI_Barrier(MPI_COMM_WORLD);
    double start = MPI_Wtime();
    for (int step = 0; step < steps; step++) {
        /* Exchange owned edge rows for neighbours' halo values. */
        MPI_Sendrecv(old+n, n, MPI_DOUBLE, up, 0,
                     old+(rows+1)*n, n, MPI_DOUBLE, down, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Sendrecv(old+rows*n, n, MPI_DOUBLE, down, 1,
                     old, n, MPI_DOUBLE, up, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        for (int y = 1; y <= rows; y++) {
            for (int x = 1; x < n-1; x++) {
                int k = y*n+x;
                next[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
            }
        }
        double *temporary = old; old = next; next = temporary;
    }
    /* The step completes only when the slowest rank finishes. */
    double elapsed = MPI_Wtime()-start, maximum;
    MPI_Reduce(&elapsed, &maximum, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);
    double *plate = NULL;
    if (rank == 0) {
        plate = calloc((size_t)n*n, sizeof(double));
        if (!plate) MPI_Abort(MPI_COMM_WORLD, 1);
        for (int x = 0; x < n; x++) plate[x] = 100.0;
    }
    /* Gather interior rows once, after the time stepping. */
    MPI_Gather(old+n, rows*n, MPI_DOUBLE, rank == 0 ? plate+n : NULL,
               rows*n, MPI_DOUBLE, 0, MPI_COMM_WORLD);
    if (rank == 0) {
        printf("mpi loop_s=%.9f\n", maximum);
        save_grid(argv[3], plate, n);
    }
    free(old); free(next); free(plate);
    MPI_Finalize();
    return 0;
}

### Compile and start two processes

```bash
module load OpenMPI/4.1.5-GCC-12.3.0
mpicc -O2 heat_mpi.c -o mpi
srun -p cpu -N1 -n2 -c1 --mem=1G --time=00:02:00 ./mpi 128 150 mpi.vtk | tee mpi-time.txt
cmp serial.vtk mpi.vtk
```

`mpicc` supplies the MPI headers and libraries. `-n2` starts two processes, each with its own arrays. We use one node here so we can learn the decomposition before studying communication across nodes.

### Checkpoint 4

There must be two ownership lines, labelled `rank 0/2` and `rank 1/2`, each with 63 interior rows. If they report `/1`, the launcher did not create the intended communicator. Stop and check the module/launcher.

`cmp` must succeed. Explain why the result can be identical even though no rank stores the complete plate while time stepping.

## 6. Combine MPI with OpenMP

The hybrid program adds the OpenMP row directive to the MPI program. Each rank still exchanges halos. Threads share only that rank's local arrays.

`MPI_Init_thread` requests `MPI_THREAD_FUNNELED`: only the original thread makes MPI calls. The MPI calls stay outside the OpenMP loop. This is a complete source file, with those changes visible rather than hidden behind compiler switches.

In [ ]:
%%writefile heat_hybrid.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>
#include <omp.h>

/* VTK adds grid dimensions so ParaView can open the result directly. */
void save_grid(const char *name, const double *t, int n) {
    FILE *f = fopen(name, "w");
    if (!f) { perror(name); exit(1); }
    fprintf(f, "# vtk DataFile Version 3.0\nHeat plate, Celsius\nASCII\n");
    fprintf(f, "DATASET STRUCTURED_POINTS\nDIMENSIONS %d %d 1\n", n, n);
    fprintf(f, "ORIGIN 0 0 0\nSPACING 1 1 1\nPOINT_DATA %d\n", n*n);
    fprintf(f, "SCALARS Temperature double 1\nLOOKUP_TABLE default\n");
    for (int k = 0; k < n*n; k++) fprintf(f, "%.17g\n", t[k]);
    if (fclose(f) != 0) exit(1);
}

int main(int argc, char **argv) {
    int rank, ranks;
    /* Only the original thread calls MPI. */
    int provided;
    MPI_Init_thread(&argc, &argv, MPI_THREAD_FUNNELED, &provided);
    if (provided < MPI_THREAD_FUNNELED) MPI_Abort(MPI_COMM_WORLD, 1);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &ranks);
    if (argc != 4) MPI_Abort(MPI_COMM_WORLD, 1);
    int n = atoi(argv[1]), steps = atoi(argv[2]);
    /* Equal blocks keep this example short: N-2 must divide by ranks. */
    if (n < 3 || n > 4096 || steps < 1 || steps > 100000 || ranks > n-2 || (n-2)%ranks != 0)
        MPI_Abort(MPI_COMM_WORLD, 1);
    int rows = (n-2)/ranks;
    double *old = calloc((size_t)(rows+2)*n, sizeof(double));
    double *next = calloc((size_t)(rows+2)*n, sizeof(double));
    if (!old || !next) MPI_Abort(MPI_COMM_WORLD, 1);
    /* Each rank has two halo rows. Rank 0's top halo is the hot edge. */
    if (rank == 0) for (int x = 0; x < n; x++) old[x] = next[x] = 100.0;
    int up = rank == 0 ? MPI_PROC_NULL : rank-1;
    int down = rank == ranks-1 ? MPI_PROC_NULL : rank+1;
    fprintf(stderr, "rank %d/%d: %d interior rows\n", rank, ranks, rows);
    MPI_Barrier(MPI_COMM_WORLD);
    double start = MPI_Wtime();
    for (int step = 0; step < steps; step++) {
        /* Exchange owned edge rows for neighbours' halo values. */
        MPI_Sendrecv(old+n, n, MPI_DOUBLE, up, 0,
                     old+(rows+1)*n, n, MPI_DOUBLE, down, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Sendrecv(old+rows*n, n, MPI_DOUBLE, down, 1,
                     old, n, MPI_DOUBLE, up, 1, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        /* Threads share this rank's arrays. MPI calls stay outside this loop. */
        #pragma omp parallel for
        for (int y = 1; y <= rows; y++) {
            for (int x = 1; x < n-1; x++) {
                int k = y*n+x;
                next[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
            }
        }
        double *temporary = old; old = next; next = temporary;
    }
    /* The step completes only when the slowest rank finishes. */
    double elapsed = MPI_Wtime()-start, maximum;
    MPI_Reduce(&elapsed, &maximum, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);
    double *plate = NULL;
    if (rank == 0) {
        plate = calloc((size_t)n*n, sizeof(double));
        if (!plate) MPI_Abort(MPI_COMM_WORLD, 1);
        for (int x = 0; x < n; x++) plate[x] = 100.0;
    }
    /* Gather interior rows once, after the time stepping. */
    MPI_Gather(old+n, rows*n, MPI_DOUBLE, rank == 0 ? plate+n : NULL,
               rows*n, MPI_DOUBLE, 0, MPI_COMM_WORLD);
    if (rank == 0) {
        printf("hybrid loop_s=%.9f\n", maximum);
        save_grid(argv[3], plate, n);
    }
    free(old); free(next); free(plate);
    MPI_Finalize();
    return 0;
}

### Run two ranks with two threads each

```bash
mpicc -O2 -fopenmp heat_hybrid.c -o hybrid
export OMP_NUM_THREADS=2
export OMP_DYNAMIC=FALSE
srun -p cpu -N1 -n2 -c2 --mem=2G --time=00:02:00 ./hybrid 128 150 hybrid.vtk | tee hybrid-time.txt
cmp serial.vtk hybrid.vtk
```

### Checkpoint 5

Check the two rank lines and the complete field comparison. Draw two rank boxes with two threads inside each. Which arrows represent shared-memory access, and which represent messages?

This hybrid run uses four active CPU workers. Our OpenMP and MPI examples use two. Their timing differences do not isolate the programming model at a fixed core count. MPI/hybrid loop time includes halo exchange; serial/OpenMP loop time does not need it. All loop timers exclude output and process startup.

## 7. Save the commands in one Slurm job

A batch file contains the same commands, plus resource requests. It lets the cluster run the exercise after you disconnect.

The file below compiles the four programs, repeats each three times and checks all fields with `cmp`. Every submission has its own results folder. It also records a separate profile and attempts ParaView post-processing. There are no build, measurement or submission helper scripts.

In [ ]:
%%writefile cpu.slurm
#!/bin/bash -l
#SBATCH --job-name=m5s1-basics
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=2
#SBATCH --cpus-per-task=2
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=cpu-%j.out
set -euo pipefail
cd "$SLURM_SUBMIT_DIR"
printf "cpu-%s.out\n" "$SLURM_JOB_ID" > latest_cpu_log.txt
N=${N:-128}; STEPS=${STEPS:-150}
module load OpenMPI/4.1.5-GCC-12.3.0
module list
# Four files, four executables. No compile-time switches choose the algorithm.
gcc -O2 heat_serial.c -o serial
gcc -O2 -fopenmp heat_openmp.c -o openmp
mpicc -O2 heat_mpi.c -o mpi
mpicc -O2 -fopenmp heat_hybrid.c -o hybrid
export OMP_NUM_THREADS=2 OMP_DYNAMIC=FALSE
export OMP_PLACES=cores OMP_PROC_BIND=close
# Save each submission in a different results folder.
out="$PWD/results/cpu-$SLURM_JOB_ID"
mkdir -p "$out"
for repeat in 1 2 3; do
    echo "Repeat $repeat" | tee -a "$out/timings.txt"
    srun -n1 -c1 ./serial "$N" "$STEPS" "$out/serial-$repeat.vtk" | tee -a "$out/timings.txt"
    srun -n1 -c2 ./openmp "$N" "$STEPS" "$out/openmp-$repeat.vtk" | tee -a "$out/timings.txt"
    srun -n2 -c2 ./mpi "$N" "$STEPS" "$out/mpi-$repeat.vtk" | tee -a "$out/timings.txt"
    srun -n2 -c2 ./hybrid "$N" "$STEPS" "$out/hybrid-$repeat.vtk" | tee -a "$out/timings.txt"
    for mode in serial openmp mpi hybrid; do
        cmp "$out/serial-1.vtk" "$out/$mode-$repeat.vtk"
        echo "PASS $mode repeat $repeat"
    done
done
# Check one known time step independently of the MPI/OpenMP comparisons.
srun -n1 -c1 ./serial 8 1 "$out/first-step.vtk"
awk 'NR>=19 && NR<=26 { expected=(NR==19 || NR==26)?0:25; if ($1!=expected) exit 1 }' "$out/first-step.vtk"
echo 'PASS first-step row: 0 25 25 25 25 25 25 0'
printf '%s\n' "$out" > latest_cpu.txt
echo CPU_PASS
# Profiling is outside the measured runs.
if command -v gprof >/dev/null; then
    gcc -O0 -g -pg heat_serial.c -o profile
    srun -n1 -c1 ./profile 512 600 "$out/profile.vtk"
    gprof ./profile gmon.out > "$out/profile.txt"
fi
# ParaView is post-processing; its status is separate from solver correctness.
if module load ParaView/5.11.2-foss-2023a; then
    export LIBGL_ALWAYS_SOFTWARE=true GALLIUM_DRIVER=llvmpipe EGL_PLATFORM=surfaceless LP_NUM_THREADS=1
    unset DISPLAY WAYLAND_DISPLAY PV_DEBUG_REMOTE_RENDERING
    ulimit -c 0
    if pvbatch --force-offscreen-rendering view.py "$N" "$out/serial-1.vtk" "$out/paraview.png"; then
        echo PARAVIEW_PASS
    else
        echo PARAVIEW_FAILED
    fi
else
    echo PARAVIEW_UNAVAILABLE
fi

### Submit and review

First save the ParaView script in the next section if you want the batch job to create an image. Then submit from the login node:

```bash
sbatch cpu.slurm
squeue -u "$USER"
```

Write down the job ID. Replace `JOBID` below with it:

```bash
sacct -j JOBID --format=JobID,State,ExitCode
cat cpu-JOBID.out
cat results/cpu-JOBID/timings.txt
```

Look for `CPU_PASS`, successful comparisons for all four modes, and three sets of measured times. Take the middle of the three times for each mode. These are small teaching runs, so expect variability.

The profile is a separate instrumented run. It is not one of the timing comparisons:

```bash
head -40 results/cpu-JOBID/profile.txt
```

Find `update`, which contains the stencil. A sampling profile may have too few samples for very short runs. Why would compiling with `-pg` change what we are timing?

## 8. Open the result in ParaView

The C programs already write a `.vtk` file. Open `serial.vtk` in desktop ParaView, click Apply, and colour by Temperature. Use the same 0 to 100 °C scale for every comparison.

For a screenshot in the batch job, save `view.py` below. This is ParaView's small Python script, not a second implementation of the solver. It opens the C result and sets the camera and colour scale. The batch file loads the ParaView module and uses `pvbatch` with CPU software rendering.

A rendering failure is reported separately from CPU correctness. It must not be presented as a successful ParaView image.

In [ ]:
%%writefile view.py
# Run with pvbatch. The solver has already written the grid in VTK format.
from pathlib import Path
import sys
from paraview.simple import (OpenDataFile, GetActiveViewOrCreate, Show, ColorBy,
                            GetColorTransferFunction, SaveScreenshot)

n = int(sys.argv[1])
field = OpenDataFile(str(Path(sys.argv[2]).resolve()))
field.UpdatePipeline()
assert field.GetDataInformation().GetNumberOfPoints() == n*n
assert field.PointData['Temperature'].GetRange(0) == (0.0, 100.0)
view = GetActiveViewOrCreate('RenderView')
image = Show(field, view)
ColorBy(image, ('POINTS', 'Temperature'))
GetColorTransferFunction('Temperature').RescaleTransferFunction(0, 100)
image.SetScalarBarVisibility(view, True)
view.CameraParallelProjection = 1
view.CameraPosition = [(n-1)/2, (n-1)/2, 2*n]
view.CameraFocalPoint = [(n-1)/2, (n-1)/2, 0]
view.CameraViewUp = [0, -1, 0]
view.CameraParallelScale = n*0.6
view.StillRender()
SaveScreenshot(sys.argv[3], view, ImageResolution=[900, 700])
print('PARAVIEW_PASS')

## 9. Optional GPU extension with OpenACC

This version has its own C file. It adds an accelerator loop to `update` and a data region around the time stepping. Both arrays remain on the GPU between steps, rather than being transferred on every update.

Read the two `acc` directives and compare them with the OpenMP version. The program checks that an NVIDIA device is available. Its loop timer excludes the initial and final data transfers.

In [ ]:
%%writefile heat_openacc.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <openacc.h>

double seconds(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec + 1e-9*t.tv_nsec;
}

/* Read only the old grid; each iteration writes a different new cell. */
void update(double *old, double *next, int n) {
    #pragma acc parallel loop present(old[0:n*n], next[0:n*n])
    for (int y = 1; y < n-1; y++) {
        for (int x = 1; x < n-1; x++) {
            int k = y*n+x;
            next[k] = 0.25*(old[k-n]+old[k+n]+old[k-1]+old[k+1]);
        }
    }
}

void save_grid(const char *name, const double *t, int n);

int main(int argc, char **argv) {
    if (argc != 4) { fprintf(stderr, "Usage: %s N STEPS output.vtk\n", argv[0]); return 1; }
    int n = atoi(argv[1]), steps = atoi(argv[2]);
    if (n < 3 || n > 4096 || steps < 1 || steps > 100000) return 1;
    double *a = calloc((size_t)n*n, sizeof(double));
    double *b = calloc((size_t)n*n, sizeof(double));
    if (!a || !b) return 1;
    /* Both grids start cold, with the same fixed hot top edge. */
    for (int x = 0; x < n; x++) a[x] = b[x] = 100.0;
    /* A queued GPU job must have a real accelerator, not a CPU fallback. */
    if (acc_get_num_devices(acc_device_nvidia) < 1) return 2;
    acc_set_device_type(acc_device_nvidia);
    /* Copy once in and once out, with both time levels resident on the GPU. */
    #pragma acc data copy(a[0:n*n], b[0:n*n])
    {
        double start = seconds();
        for (int step = 0; step < steps; step++) {
            /* Alternate buffers; no whole-grid copy is needed. */
            double *old = step % 2 == 0 ? a : b;
            double *next = step % 2 == 0 ? b : a;
            update(old, next, n);
        }
        printf("openacc loop_s=%.9f\n", seconds()-start);
    }
    double *final = steps % 2 == 0 ? a : b;
    printf("Cell below hot edge: %.9f C\n", final[n+1]);
    save_grid(argv[3], final, n);
    free(a); free(b);
    return 0;
}

/* VTK adds grid dimensions so ParaView can open the result directly. */
void save_grid(const char *name, const double *t, int n) {
    FILE *f = fopen(name, "w");
    if (!f) { perror(name); exit(1); }
    fprintf(f, "# vtk DataFile Version 3.0\nHeat plate, Celsius\nASCII\n");
    fprintf(f, "DATASET STRUCTURED_POINTS\nDIMENSIONS %d %d 1\n", n, n);
    fprintf(f, "ORIGIN 0 0 0\nSPACING 1 1 1\nPOINT_DATA %d\n", n*n);
    fprintf(f, "SCALARS Temperature double 1\nLOOKUP_TABLE default\n");
    for (int k = 0; k < n*n; k++) fprintf(f, "%.17g\n", t[k]);
    if (fclose(f) != 0) exit(1);
}

In [ ]:
%%writefile gpu.slurm
#!/bin/bash -l
#SBATCH --job-name=m5s1-openacc
#SBATCH --partition=gpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --gres=gpu:1
#SBATCH --mem=2G
#SBATCH --time=00:06:00
#SBATCH --output=gpu-%j.out
set -euo pipefail
cd "$SLURM_SUBMIT_DIR"
printf "gpu-%s.out\n" "$SLURM_JOB_ID" > latest_gpu_log.txt
N=${N:-128}; STEPS=${STEPS:-150}
module load nvhpc/25.7
module list
nvidia-smi
nvc -O2 -acc -gpu=ccnative -Minfo=accel heat_openacc.c -o openacc
# Make a CPU reference with exactly the same inputs in this job.
gcc -O2 heat_serial.c -o serial
out="$PWD/results/gpu-$SLURM_JOB_ID"
mkdir -p "$out"
./serial "$N" "$STEPS" "$out/reference.vtk"
./openacc "$N" "$STEPS" "$out/openacc.vtk"
# Allow tiny rounding differences, comparing all scalar values after the VTK header.
paste "$out/reference.vtk" "$out/openacc.vtk" | awk '
NR>10 { difference=$1-$2; if(difference<0) difference=-difference;
        if(difference>maximum) maximum=difference }
END { print "Maximum GPU error:",maximum+0; exit(maximum>1e-10) }'
printf '%s\n' "$out" > latest_gpu.txt
echo GPU_PASS

Submit from the login node:

```bash
module avail nvhpc
sbatch gpu.slurm
squeue -u "$USER"
```

### Checkpoint 6

After completion, inspect `gpu-JOBID.out`. Look for the compiler's offload information, the GPU shown by `nvidia-smi`, a small maximum field error and `GPU_PASS`. A queued job has not run. The script compares every temperature with a CPU reference using the same inputs, allowing `1e-10` for rounding differences.

Why might this small grid be slower on a GPU? Which costs are excluded from the reported loop time?

The [extended notebook](M5S1_hybrid_computing_extended.ipynb) contains the longer CUDA and library exercises if you want to continue beyond these basics.

## 10. Try a larger experiment

Predict what will change, then submit the cases one at a time:

```bash
N=512 STEPS=150 sbatch cpu.slurm
N=512 STEPS=600 sbatch cpu.slurm
```

Record each job ID. Every job keeps its files in `results/cpu-JOBID`, so the new measurements do not overwrite earlier ones. If testing the GPU, pass the same `N` and `STEPS` to `gpu.slurm`.

The larger grid has about 16 times as many interior cells. Four times as many steps means four times as many updates. Larger loops can make worker overhead less dominant, but speedup is not guaranteed.

Grid spacing stays at one, so the larger grid represents a larger plate. This is a workload-size experiment, not refinement of the same plate or strong scaling.

Hand in the timings, one image, the job IDs and short answers:

1. Which implementation helped at each workload size?
2. Which comparisons used the same number of workers?
3. Where does each implementation move data?
4. What did the checkpoints establish before you compared speed?
5. What would you test next, and why?

---
## Reproduce the exercise from Jupyter

The source cells above write the same files that students create in SSH. The function below only sends a command to a shell and prints its output. It does not calculate temperatures, compare fields or implement a scheduler.

The commands that follow are the same `sbatch`, `cat` and `sacct` commands used above. `sbatch --wait` waits for the submitted job to finish. Each run submits a new job, whose ID appears in the output. GPU jobs may stay queued until a device is available. Keep the SciTech kernel connected while waiting.

In [ ]:
import subprocess

def run(command):
    print('$ ' + command, flush=True)
    environment = os.environ.copy()
    for key in ('SLURM_MEM_PER_CPU', 'SLURM_MEM_PER_GPU', 'SLURM_MEM_PER_NODE'):
        environment.pop(key, None)
    process = subprocess.Popen(['bash', '-lc', command], env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Command failed; inspect the output above.')

In [ ]:
run('sbatch --wait cpu.slurm')

In [ ]:
run('cat "$(cat latest_cpu_log.txt)"; result=$(cat latest_cpu.txt); if [ -f "$result/profile.txt" ]; then head -30 "$result/profile.txt"; fi')

In [ ]:
# A GPU failure should remain visible and should not hide the CPU image.
try:
    run('sbatch --wait gpu.slurm')
except RuntimeError:
    print('GPU calculation did not pass. Inspect the GPU job log.')
run('if [ -f latest_gpu_log.txt ]; then cat "$(cat latest_gpu_log.txt)"; fi')

### The ParaView image

This displays only the PNG produced by the completed ParaView job. If rendering failed, inspect the CPU log for `PARAVIEW_FAILED` or `PARAVIEW_UNAVAILABLE`; the CPU comparison results remain separate.

In [ ]:
from IPython.display import Image, display

result = Path('latest_cpu.txt').read_text().strip()
image = Path(result) / 'paraview.png'
if image.exists() and image.read_bytes().startswith(b'\x89PNG\r\n\x1a\n'):
    display(Image(filename=str(image)))
else:
    print('No ParaView image was produced. Read the rendering status in the CPU job log.')